In [ ]:
# pip install voila ipywidgets reportlab pillow
#
# How scoring works:
# 1. Each indicator score (1-4) is answered directly by the user.
# 2. A component score is the plain average of its indicators.
# 3. A layer score is the weighted average of its components (using the
#    component weight sliders).
# 4. The overall score is the weighted average of the three layer scores
#    (using the layer weight sliders).
#
# Radar plots now also use these weighted values:
# - The component radar plot still shows the raw indicator scores (there is
#   no weight slider at the indicator level).
# - The layer radar plot shows each component score multiplied by its own
#   weight slider (capped at 4), so a heavier weight makes that arm bigger.
# - The overall radar plot shows each layer score multiplied by its own
#   weight slider (capped at 4), same idea.

In [ ]:
import io
import csv
import base64
import textwrap
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from PIL import Image as PILImage
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    Table, TableStyle, HRFlowable, PageBreak, KeepTogether
)

In [ ]:
LEVEL_NAMES = ["Initial", "Basic", "Advanced", "Integrated"]

display(HTML("""
<style>
.qcard { border:1px solid #e5e7eb; border-radius:10px; padding:16px 18px;
         margin-bottom:14px; background:#ffffff; box-shadow:0 1px 3px rgba(0,0,0,0.05); }
.layer-header { padding:22px 26px; border-radius:12px; margin:26px 0 14px 0;
                box-shadow:0 6px 18px rgba(15,23,42,0.07); font-family:Arial, sans-serif; }
.layer-header h2 { margin:0; font-size:26px; }
.layer-header p { margin:8px 0 0; font-size:14px; }
.widget-accordion .p-Accordion-child { border-radius:8px !important; }
.weight-box { border:1px dashed #d1d5db; border-radius:8px; padding:8px 12px;
              margin:0 0 10px 0; background:#f9fafb; }
</style>
"""))

# ALL_WIDGETS stores every question. key = "Layer::Component::Indicator"
ALL_WIDGETS = {}

# COMPONENT_WEIGHTS stores one slider per component. key = "Layer::Component"
COMPONENT_WEIGHTS = {}

# LAYER_WEIGHTS stores one slider per layer. key = "Layer"
LAYER_WEIGHTS = {}


def make_question(layer_name, component_name, indicator, description, question_text, levels, widget_store):
    options = []
    for i in range(4):
        option_text = "(" + LEVEL_NAMES[i] + ") " + " - " + levels[i]
        options.append(option_text)

    radio = widgets.RadioButtons(
        options=options,
        layout=widgets.Layout(width='100%'),
        style={'description_width': '0px'}
    )

    key = layer_name + "::" + component_name + "::" + indicator
    widget_store[key] = {
        "layer": layer_name,
        "component": component_name,
        "indicator": indicator,
        "description": description,
        "question": question_text,
        "widget": radio
    }

    header_html = (
        "<div style='margin-bottom:8px;'>"
        "<span style='font-weight:700;color:#111827;font-size:15px;'>" + indicator + "</span><br>"
        "<span style='color:#6b7280;font-size:13px;'>" + description + "</span><br>"
        "<span style='color:#4b5563;font-size:13.5px;font-style:italic;'>" + question_text + "</span>"
        "</div>"
    )
    header = widgets.HTML(header_html)

    box = widgets.VBox([header, radio])
    box.add_class('qcard')
    return box


def make_weight_slider(label_text, key, weight_store, color="#111827", default=0.5):
    """Create a 0-1 weighting-factor slider and register it in weight_store."""
    slider = widgets.FloatSlider(
        value=default, min=0.0, max=1.0, step=0.01,
        readout_format='.2f',
        layout=widgets.Layout(width='260px'),
        style={'handle_color': color}
    )
    weight_store[key] = slider

    label_html = "<span style='font-size:12.5px;font-weight:600;color:" + color + ";'> Weight &mdash; " + label_text + "</span>"
    label = widgets.HTML(label_html)

    row = widgets.HBox([label, slider], layout=widgets.Layout(align_items='center', gap='10px'))
    row.add_class('weight-box')
    return row


def build_component(layer_name, component_name, indicators, widget_store, comp_weight_store, accent):
    weight_key = layer_name + "::" + component_name
    weight_row = make_weight_slider(component_name, weight_key, comp_weight_store, color=accent)

    boxes = [weight_row]
    for indicator, description, question_text, levels in indicators:
        box = make_question(layer_name, component_name, indicator, description, question_text, levels, widget_store)
        boxes.append(box)

    inner = widgets.VBox(boxes, layout=widgets.Layout(padding='6px 2px'))
    acc = widgets.Accordion(children=[inner], selected_index=None)
    title = component_name + "  -  " + str(len(indicators)) + " indicators"
    acc.set_title(0, title)
    return acc


def build_layer(layer_name, subtitle, components, accent, bg, widget_store, comp_weight_store, layer_weight_store):
    header_html = (
        "<div class='layer-header' style='background:" + bg + ";border-left:7px solid " + accent + ";'>"
        "<h2 style='color:" + accent + ";'>" + layer_name + " Layer</h2>"
        "<p style='color:#4b5563;'>" + subtitle + "</p></div>"
    )
    header = widgets.HTML(header_html)

    layer_weight_row = make_weight_slider(
        layer_name + " Layer (contribution to overall score)",
        layer_name, layer_weight_store, color=accent
    )

    section_children = [header, layer_weight_row]
    for name, items in components.items():
        comp_widget = build_component(layer_name, name, items, widget_store, comp_weight_store, accent)
        section_children.append(comp_widget)

    return widgets.VBox(section_children, layout=widgets.Layout(margin='0 0 6px 0'))

In [ ]:
intro_header = widgets.HTML(
    """
    <div class="layer-header" style="background:linear-gradient(135deg,#f8fafc 0%,#edf2f7 100%);
                border-left:7px solid #111827;">
        <h2 style="color:#111827;">Manufacturing Infrastructure Maturity Assessment</h2>
        <p style="color:#4b5563;max-width:760px;line-height:1.6;">
            Rate each readiness indicator across the Physical, Digital and Organizational layers.
            Expand a category, read the assessment question, then choose the maturity level
            (1&ndash;4) that best matches current practice.
        </p>
    </div>
    """
)

resp_name = widgets.Text(placeholder='Full name', layout=widgets.Layout(width='320px'))
resp_facility = widgets.Text(placeholder='Facility / Fab lab name', layout=widgets.Layout(width='320px'))
resp_role = widgets.Text(placeholder='Role (e.g., Manager, Assessor)', layout=widgets.Layout(width='320px'))

resp_box = widgets.VBox([
    widgets.HTML("<b style='color:#111827;'>Respondent name</b>"), resp_name,
    widgets.HTML("<b style='color:#111827;'>Facility / Fab lab name</b>"), resp_facility, # this is one
    widgets.HTML("<b style='color:#111827;'>Role</b>"), resp_role,
], layout=widgets.Layout(gap='6px', margin='0 0 10px 0'))
resp_box.add_class('qcard');

In [ ]:
PHYSICAL_DATA = {
    "Space use": [  # Infrastructure components
        ("Space adaptability and versatility",  # Readiness indicator
         "Capacity to accommodate diversified activities (e.g., modular design & configurability, interchangeable zones).",  # Description
         "How easily can the layout accommodate diversified activities like modular design and configuration?",  # Main Assessment Question
         ["Static layout; activities overlap without clear separation.",
          "Limited zoning; some movement of furniture possible.",
          "Dedicated, interchangeable zones for different project types.",
          "Fully modular and reconfigurable space for diverse parallel workflows."]),  # options
        ("Space accessibility",
         "Physical access to the factories and the availability of opportunities for experimentation and collaboration.",
         "How inclusive and barrier-free is physical access for professionals, artists, and hobbyists?",
         ["Highly restricted or exclusive access.",
          "Open during limited hours with manual supervision.",
          "Flexible access with digital booking systems in place.",
          "Inclusive access with 24/7 clear wayfinding and open lab policies."]),
        ("Urban integration",
         "Proximity to densely populated areas with low emission (noise, dust emission, weight, space, safety...).",
         "To what extent does the facility operate without causing urban disruption (noise, dust, safety)?",
         ["High disruption; frequent complaints or safety risks.",
          "Basic filters/housing; occasional noise issues.",
          "Controlled emissions with moderate architectural noise control.",
          "Seamlessly integrated; low-emission machinery and active community engagement."]),
    ],
    "Machines & equipements": [
        ("Multi-functionality / versatility",
         "Easily go from prototyping to production (high grade of multi-functionality: e.g., automated material handling unit integrated, automated testing unit integrated).",
         "Do machines support rapid transitions from prototyping to small-series production?",
         ["Single-purpose machines with long changeover times.",
          "Low multi-functionality; manual tool changes required.",
          "Moderate versatility; some automated tool changers or modularity.",
          "High versatility; hybrid equipment (e.g., CNC with ATC) and integrated material handling."]),
        ("Machine accessibility",
         "Highly intuitive, easy for all users, UX of machines. Prioritise acquiring user-friendly, state-of-the-art machinery that are accessible to all stakeholders.",
         "How intuitive and user-friendly is the machine operation for users with varying skill levels?",
         ["Complex interfaces; requires constant expert oversight.",
          "Standard operation requiring multi-day training.",
          "User-friendly HMIs with guided job setups and checklists.",
          "Highly intuitive; standardized procedures allowing rapid beginner onboarding."]),
        ("Openess of machine",
         "Machines are detailed documented, with fully independent repairability and modification.",
         "Are machines documented and modifiable for independent repair?",
         ["Proprietary systems; repairs strictly limited to external vendors.",
          "Partially documented; some local troubleshooting possible.",
          "Well-documented commercial machines with local spare parts.",
          "Open-source hardware; fully independent repairability and modification."]),
    ],
    "Operations": [
        ("Material handling",
         "Storage for materials and finished products, with easy access to manage production flows.",
         "How efficiently are raw materials and finished products stored and moved?",
         ["No system; materials stored in walkways causing congestion.",
          "Ad-hoc shelving and manual movement.",
          "Organized vertical storage and labeled staging areas.",
          "Automated retrieval or structured flow minimizing space usage."]),
        ("Maintenance & internal capacity for reliability",
         "Machines are maintained proactively by in-house teams, minimizing breakdowns and external interventions. The space is equipped to support routine upkeep and minor repairs, ensuring machines stay reliable and operations remain uninterrupted.",
         "How proactively are machines maintained by in-house teams to minimize downtime?",
         ["Reactive; machines repaired only after breakdown.",
          "Irregular manual checks by staff.",
          "Structured preventive cycles (weekly/monthly routines).",
          "Data-based predictive maintenance using real-time machine tracking."]),
        ("Equipment reconfiguration",
         "Workstations are adapted to the different stages of production, from concept to finished product.",
         "How quickly can workstations be adapted to different stages of production?",
         ["Fixed workstations; reconfiguration is impossible.",
          "Mobile benches requiring significant manual effort.",
          "Modular fixtures with quick-connect utilities.",
          "Agile production lines that retool rapidly for customized orders."]),
        ("Material management",
         "Sourcing services and easy access provision to common materials for users.",
         "How systematic is the provision and sourcing of common materials for end-users?",
         ["Users must source all their own materials externally.",
          "Ad-hoc local stock of standard inputs like filaments.",
          "Centralized access to common stock with local supplier ties.",
          "Integrated supply network ensuring 24/7 material availability."]),
        ("Adaption and flexibility",
         "High degree of product change, end-user experience and co-creation with the customer in order to adjust the product design and manufacturing process for small-batch production, with adjustable production lines.",
         "To what degree can production adjust based on user feedback and co-creation?",
         ["Rigid processes; unable to handle custom orders.",
          "Occasional manual adjustments for unique projects.",
          "Moderate flexibility for small batches (1-30 units).",
          "High degree of co-creation; adjustable lines for small-batch production."]),
    ],
    "Energy efficiency": [
        ("Renewable Energy sources",
         "Renewable energy sources are self-generated or purchased.",
         "Is the facility powered by self-generated or purchased renewable energy?",
         ["Entirely dependent on non-renewable grid electricity.",
          "Minor behavioral energy saving (e.g., turning off hoods).",
          "Certified renewable energy purchased from providers.",
          "Significant on-site generation (e.g., rooftop solar/wind)."]),
        ("Energy efficiency",
         "Energy efficiency methods like energy tracking or reduction of energy consumption are applied.",
         "Are methods like energy tracking or machine-level consumption reduction applied?",
         ["No tracking or efficiency measures in place.",
          "Ad-hoc monitoring at the building level.",
          "Machine-level monitoring (e.g., Shelly plugs) implemented.",
          "Real-time dashboards for load optimization and idle-time shutdowns."]),
    ],
}

physical_section = build_layer(
    "Physical",
    "Space, machines, operations and energy readiness of the facility.",
    PHYSICAL_DATA, accent="#c0392b", bg="#fdecea", widget_store=ALL_WIDGETS,
    comp_weight_store=COMPONENT_WEIGHTS, layer_weight_store=LAYER_WEIGHTS
)

In [ ]:
DIGITAL_DATA = {
    "Digital tools": [
        ("Integration",
         "High integration of digital tools across operations.",
         "How seamlessly are tools like CAD/CAM/MES connected across operations?",
         ["Isolated tools; all data transfer is manual.",
          "Partial integration via shared file repositories.",
          "Integrated toolchains for core manufacturing processes.",
          "Full data flow integration from design to machine activation."]),
        ("Automation",
         "High automated digital workflows across manufacturing operations.",
         "What level of automation exists in the digital workflows (e.g., file-sharing, toolpaths)?",
         ["All steps are manual (backups, logging, sharing).",
          "Basic automation like automated data backups.",
          "Partially automated workflows (e.g., machine usage logs).",
          "Highly automated process chains with minimal human intervention."]),
        ("Accessibility",
         "Easy to use digital tools for manufacturing operations.",
         "Are digital tools intuitive enough for users with varying technical expertise?",
         ["Expert-level software only; high cost/learning curve.",
          "Standard software requiring multi-day training.",
          "Inclusive onboarding with tutorials and modular training.",
          "Low-barrier interfaces with embedded \"how-to\" guides."]),
        ("Openness",
         "High grade of free and open-source software tools.",
         "To what extent are free and open-source software (FOSS) tools utilized?",
         ["Exclusively proprietary software.",
          "Mostly proprietary with isolated open-source tools (e.g., Slicer).",
          "Significant use of FOSS in core workflows.",
          "Entirely open-source infrastructure with active upstream contributions."]),
        ("Adaption and flexibility",
         "High grade of interoperability of digital tools and files.",
         "How interoperable are digital tools and design files across the network?",
         ["Proprietary file locks; no interoperability.",
          "Manual file repairs required for design transfers.",
          "Cloud-based exchange of standardized file formats.",
          "Full interoperability; seamless multi-site collaboration without data loss."]),
    ],
    "Data Management": [
        ("Data sharing",
         "Pro-active sharing of data and information.",
         "How proactively is project and process information shared?",
         ["No data sharing; information silos exist.",
          "Informal sharing via ad-hoc chats or emails.",
          "Searchable digital repositories with structured sharing.",
          "Federated network participation with real-time information sharing."]),
        ("Storage & File System",
         "Pro-active integration of structured data management and use of digital repositories.",
         "How structured is the data management and use of centralized repositories?",
         ["Unstructured local saving on machine PCs.",
          "Convenience-based cloud storage (e.g., Google Drive) without protocols.",
          "Centralized repositories with version control (e.g., Git-based).",
          "Structured storage integrated with Digital Product Passports."]),
    ],
    "Data Monitoring": [
        ("Process Tracking",
         "Automated tracking of resource and machine metrics for reporting and evaluation of sustainable manufacturing methods.",
         "Is there automated tracking of machine metrics for sustainability evaluation?",
         ["No tracking; performance is not measured.",
          "Manual logging of machine usage times.",
          "IoT-based automated tracking of machine runtime and state.",
          "Full lifecycle data tracking for DPP generation and reporting."]),
        ("Network Connectivity",
         "Local (wireless) network existent with active internet connection.",
         "How reliable is the local factory network and its internet connection?",
         ["Unstable or no internet; no machine connectivity.",
          "Basic Wi-Fi for standard office use.",
          "Reliable Wi-Fi with machine access to local networks.",
          "High-speed, secure network enabling remote machine control."]),
    ],
}

digital_section = build_layer(
    "Digital",
    "Digital tools, data management and monitoring readiness of the facility.",
    DIGITAL_DATA, accent="#0e7490", bg="#e6fbfb", widget_store=ALL_WIDGETS,
    comp_weight_store=COMPONENT_WEIGHTS, layer_weight_store=LAYER_WEIGHTS
)

In [ ]:
ORGANIZATIONAL_DATA = {
    "Manufacturing structure": [
        ("Distributed Governance",
         "High grade of autonomous manufacturing activities.",
         "To what extent is autonomous decision-making delegated to teams?",
         ["Centralized control; all decisions made by management.",
          "Partial autonomy within strict hierarchical rules.",
          "High grade of autonomy for teams within project streams.",
          "Fully decentralized, participatory governance models."]),
        ("Inter-organizational collaboration",
         "Having a strong internal and external network capacity and reliance.",
         "How strong is the internal and external network of partners (SMEs, artists, civic actors)?",
         ["No external partnerships; facility operates in isolation.",
          "Occasional collaboration on specific events.",
          "Regular involvement of external actors in manufacturing projects.",
          "Dense, resilient network of regional socio-economic partners."]),
    ],
    "Project planning and coordination": [
        ("Integration",
         "Long-term strategic planning of project/operations timeline.",
         "Is project planning aligned with long-term strategic objectives?",
         ["No planning; reactive response to daily requests.",
          "Short-term scheduling of bookings and tasks.",
          "Operational timelines aligned with factory development goals.",
          "Strategic integration of manufacturing capacity with urban needs."]),
        ("Information sharing",
         "Proactive information and knowledge access and sharing.",
         "How proactive is the access to knowledge and operational data?",
         ["Knowledge held by key individuals only.",
          "Information available upon specific request.",
          "Searchable digital platforms with proactive project updates.",
          "Open communities of practice with shared technical resources."]),
        ("Processes adaption and flexibility (changeover ability)",
         "High integrated processes with high ad-hoc decision-making capability.",
         "What is the capability for high ad-hoc decision-making in production?",
         ["Rigid schedules; unable to reprioritize projects.",
          "Periodic adjustment of slots for urgent tasks.",
          "Rapid reconfiguration of priorities in response to demand shifts.",
          "Highly integrated processes with fluid ad-hoc scaling."]),
    ],
    "Supply chain management": [
        ("Local supply chain",
         "Small-scale manufacturing dedicated to fulfilling local needs, in a flexible supply network of regional socio-economic actors using local resources (local 50 km, regional 500 km, continental 1800 km, global 8000 km).",
         "To what extent are resources sourced from local socio-economic actors?",
         ["Global sourcing only; no local focus.",
          "Occasional use of local suppliers for standard items.",
          "Majority of materials sourced within 50-500 km range.",
          "Fully integrated into urban metabolism using local/recycled resources."]),
        ("Stakeholders' integration",
         "Stakeholder engagement in distributed design and open production networks that are co-located and community-driven.",
         "Are relevant actors involved in co-design and open production?",
         ["Top-down production without stakeholder input.",
          "Feedback collected via surveys after production.",
          "Active stakeholder engaging in co-location and joint planning.",
          "Community-driven innovation with integrated multi-actor co-design."]),
        ("Circularity",
         "Integrate principles of the circular economy into all stages of production, prioritising material reuse, waste reduction, and sustainable sourcing from local suppliers to enhance resource efficiency.",
         "How deeply are circular economy principles integrated into all production stages?",
         ["Linear model; no focus on reuse or waste.",
          "Basic waste sorting and recycling practices.",
          "Active material reuse and upcycling strategies in place.",
          "Circularity as the core business model for all product lifecycles."]),
    ],
    "Quality control and process improvement": [
        ("Quality measurement system",
         "Presence of a quality monitoring and measurement system (\"certified\").",
         "Is there a presence of formal quality monitoring or certified protocols?",
         ["No quality control; frequent errors.",
          "Informal peer checks and ad-hoc feedback.",
          "Internally documented protocols and performance indicators.",
          "Formalized, data-driven quality assurance (e.g., ISO-aligned)."]),
    ],
    "Knowledge management": [
        ("Knowledge transfer",
         "Training programmes to equip workers and collaborators with the skills required to operate advanced technologies. Emphasise upskilling initiatives to close existing skill gaps and enhance the factory's technological capacity.",
         "How effective are training programs and upskilling initiatives?",
         ["No systematic knowledge transfer.",
          "Informal peer-to-peer mentoring on the shop floor.",
          "Structured onboarding and training modules for machines.",
          "Comprehensive knowledge transfer systems and expert pools."]),
    ],
}

organizational_section = build_layer(
    "Organizational",
    "Governance, planning, supply chain, quality and knowledge readiness of the facility.",
    ORGANIZATIONAL_DATA, accent="#6d28d9", bg="#f1edfd", widget_store=ALL_WIDGETS,
    comp_weight_store=COMPONENT_WEIGHTS, layer_weight_store=LAYER_WEIGHTS
)

In [ ]:
LAYER_COLORS = {"Physical": "#c0392b", "Digital": "#0e7490", "Organizational": "#6d28d9"}


def _wrap_label(label, width=14):
    """Wrap a long label onto multiple lines so it fits nicely on the radar chart."""
    wrapped_lines = textwrap.wrap(label, width=width, break_long_words=False)
    if len(wrapped_lines) == 0:
        return label
    return "\n".join(wrapped_lines)


def plot_radar(labels, values, title, color="#2563eb", max_val=4, figsize=(4.6, 4.6)):
    n = len(labels)

    angles = []
    for i in range(n):
        angle = i / n * 2 * np.pi
        angles.append(angle)
    angles.append(angles[0])

    vals = list(values)
    vals.append(values[0])

    wrapped_labels = []
    for label in labels:
        wrapped_labels.append(_wrap_label(label))

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(polar=True))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, max_val)
    ax.set_rlabel_position(0)

    # y-axis ticks scale to max_val instead of always being 1,2,3,4.
    tick1 = max_val * 0.25
    tick2 = max_val * 0.50
    tick3 = max_val * 0.75
    tick4 = max_val
    ax.set_yticks([tick1, tick2, tick3, tick4])
    tick_labels = [format(tick1, ".1f"), format(tick2, ".1f"), format(tick3, ".1f"), format(tick4, ".1f")]
    ax.set_yticklabels(tick_labels, fontsize=8, color="#9ca3af")
    ax.grid(color="#e5e7eb", linewidth=0.8, linestyle="-")
    ax.spines["polar"].set_color("#d1d5db")
    ax.spines["polar"].set_linewidth(1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(wrapped_labels, fontsize=9, color="#1f2937")
    ax.tick_params(axis="x", pad=14)

    fill_x = np.linspace(0, 2 * np.pi, 100)
    ax.fill_between(fill_x, 0, max_val / 2, color="#f3f4f6", alpha=0.4, zorder=0)

    ax.plot(angles, vals, color=color, linewidth=2.2, marker="o", markersize=4,
            markerfacecolor="white", markeredgecolor=color, markeredgewidth=1.5, zorder=3)
    ax.fill(angles, vals, color=color, alpha=0.22, zorder=2)

    ax.set_title(title, fontsize=12.5, fontweight="bold", color="#111827", pad=26)

    fig.tight_layout(pad=1.5)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def compute_scores(widget_store, comp_weight_store, layer_weight_store):
    """Turn each answered indicator level into component scores and weighted layer scores."""
    indicator_levels = {}
    for item in widget_store.values():
        layer = item["layer"]
        component = item["component"]
        indicator = item["indicator"]
        widget = item["widget"]

        if widget.index is not None:
            level = widget.index + 1
        else:
            level = 0

        if layer not in indicator_levels:
            indicator_levels[layer] = {}
        if component not in indicator_levels[layer]:
            indicator_levels[layer][component] = {}
        indicator_levels[layer][component][indicator] = level

    scores = {}
    for layer in indicator_levels:
        comps = indicator_levels[layer]
        comp_scores = {}
        comp_weights_used = {}

        for component in comps:
            inds = comps[component]
            if len(inds) == 0:
                continue

            total = 0
            for level in inds.values():
                total = total + level
            comp_scores[component] = total / len(inds)

            weight_key = layer + "::" + component
            if weight_key in comp_weight_store:
                comp_weights_used[component] = comp_weight_store[weight_key].value
            else:
                comp_weights_used[component] = 0.5

        if len(comp_scores) == 0:
            continue

        weight_sum = 0
        for w in comp_weights_used.values():
            weight_sum = weight_sum + w

        weighted_total = 0
        for component in comp_scores:
            weighted_total = weighted_total + comp_scores[component] * comp_weights_used[component]

        if weight_sum > 0:
            layer_score = weighted_total / weight_sum
        else:
            layer_score = 0.0

        scores[layer] = {
            "components": comp_scores,
            "component_weights": comp_weights_used,
            "layer_score": layer_score,
        }

    return indicator_levels, scores


def build_all_radars(widget_store, comp_weight_store, layer_weight_store):
    """Build every radar chart. Component radars stay raw (no indicator-level weight).
    Layer and overall radars use weighted values (score x its own weight slider), and
    the chart's axis auto-scales to the largest weighted value instead of clipping at 4,
    so a heavily-weighted arm stays visually distinguishable."""
    if len(widget_store) == 0:
        raise ValueError("No questions found in ALL_WIDGETS - build the form cells before submitting.")

    indicator_levels, scores = compute_scores(widget_store, comp_weight_store, layer_weight_store)
    radar_bytes = {"components": {}, "layers": {}, "overall": None}

    # Component radar charts: raw indicator scores, fixed 0-4 scale.
    for layer in indicator_levels:
        comps = indicator_levels[layer]
        color = LAYER_COLORS.get(layer, "#2563eb")
        for component in comps:
            inds = comps[component]
            if len(inds) == 0:
                continue
            labels = list(inds.keys())
            values = list(inds.values())
            key = layer + "::" + component
            radar_bytes["components"][key] = plot_radar(labels, values, component, color, figsize=(4.2, 4.2))

    # Layer radar charts: each component score x its own weight slider, axis auto-scales.
    for layer in scores:
        data = scores[layer]
        color = LAYER_COLORS.get(layer, "#2563eb")
        labels = list(data["components"].keys())
        values = []
        for component in labels:
            raw_score = data["components"][component]
            weight = data["component_weights"][component]
            weighted_value = raw_score * weight
            values.append(weighted_value)

        chart_max = 4
        for v in values:
            if v > chart_max:
                chart_max = v

        title = layer + " Layer"
        radar_bytes["layers"][layer] = plot_radar(labels, values, title, color, max_val=chart_max, figsize=(5, 5))

    # Overall radar chart: each layer score x its own weight slider, axis auto-scales.
    layer_names = list(scores.keys())
    layer_weights_used = {}
    for layer in layer_names:
        if layer in layer_weight_store:
            layer_weights_used[layer] = layer_weight_store[layer].value
        else:
            layer_weights_used[layer] = 0.5

    overall_values = []
    for layer in layer_names:
        raw_score = scores[layer]["layer_score"]
        weight = layer_weights_used[layer]
        weighted_value = raw_score * weight
        overall_values.append(weighted_value)

    if len(layer_names) > 0:
        overall_chart_max = 4
        for v in overall_values:
            if v > overall_chart_max:
                overall_chart_max = v

        radar_bytes["overall"] = plot_radar(
            layer_names, overall_values, "Technical Infrastructure Overview", "#111827",
            max_val=overall_chart_max, figsize=(5.4, 5.4)
        )

    # Weighted overall score shown as text (uses the raw layer scores, weighted by layer weight).
    weight_sum = 0
    for w in layer_weights_used.values():
        weight_sum = weight_sum + w

    weighted_total = 0
    for layer in layer_names:
        weighted_total = weighted_total + scores[layer]["layer_score"] * layer_weights_used[layer]

    if len(layer_names) > 0 and weight_sum > 0:
        weighted_overall = weighted_total / weight_sum
    else:
        weighted_overall = 0.0

    scores["_overall"] = {"layer_weights": layer_weights_used, "weighted_overall_score": weighted_overall}

    return scores, radar_bytes

In [ ]:
LAYER_ACCENTS = {"Physical": "#c0392b", "Digital": "#0e7490", "Organizational": "#6d28d9"}


def make_safe_filename_part(text):
    """Turn free text into a filename-safe string (letters, numbers, spaces, - and _ only)."""
    if not text:
        text = "unknown"

    safe_chars = ""
    for c in text:
        if c.isalnum() or c == " " or c == "_" or c == "-":
            safe_chars = safe_chars + c

    safe_chars = safe_chars.strip()
    safe_chars = safe_chars.replace(" ", "_")
    return safe_chars


def _scaled_image(img_bytes, max_width, max_height):
    """Fit an image into a bounding box while preserving aspect ratio. Returns None if no image."""
    if not img_bytes:
        return None
    pil_img = PILImage.open(io.BytesIO(img_bytes))
    w, h = pil_img.size
    ratio = min(max_width / w, max_height / h)
    return RLImage(io.BytesIO(img_bytes), width=w * ratio, height=h * ratio)


def group_widgets_by_layer_and_component(widget_store):
    grouped = {}
    for item in widget_store.values():
        layer = item["layer"]
        component = item["component"]
        if layer not in grouped:
            grouped[layer] = {}
        if component not in grouped[layer]:
            grouped[layer][component] = []
        grouped[layer][component].append(item)
    return grouped


def generate_pdf(meta, widget_store, radar_bytes, scores, base_filename):
    filename = base_filename + ".pdf"

    doc = SimpleDocTemplate(
        filename, pagesize=letter,
        topMargin=54, bottomMargin=54, leftMargin=54, rightMargin=54,
        title="Facility Maturity Self-Assessment Report"
    )
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(name='ReportTitle', fontSize=22, leading=26,
                               textColor=colors.HexColor('#111827'), fontName='Helvetica-Bold',
                               alignment=TA_CENTER, spaceAfter=4))
    styles.add(ParagraphStyle(name='ReportSubtitle', fontSize=10.5, leading=14,
                               textColor=colors.HexColor('#6b7280'), fontName='Helvetica',
                               alignment=TA_CENTER, spaceAfter=18))
    styles.add(ParagraphStyle(name='LayerHeading', fontSize=17, spaceBefore=22, spaceAfter=14,
                               textColor=colors.white, fontName='Helvetica-Bold',
                               backColor=colors.HexColor('#111827'),
                               leftIndent=8, borderPadding=(8, 8, 8, 8)))
    styles.add(ParagraphStyle(name='LayerSubinfo', fontSize=9.5, spaceBefore=2, spaceAfter=10,
                               textColor=colors.HexColor('#6b7280'), fontName='Helvetica-Oblique'))
    styles.add(ParagraphStyle(name='CompHeading', fontSize=12.5, spaceBefore=12, spaceAfter=2,
                               textColor=colors.HexColor('#111827'), fontName='Helvetica-Bold'))
    styles.add(ParagraphStyle(name='CompWeight', fontSize=8.7, spaceAfter=6,
                               textColor=colors.HexColor('#9333ea'), fontName='Helvetica-Oblique'))
    styles.add(ParagraphStyle(name='QText', fontSize=10, spaceAfter=1, leading=13,
                               textColor=colors.HexColor('#1f2937'), fontName='Helvetica-Bold'))
    styles.add(ParagraphStyle(name='DescText', fontSize=8.7, spaceAfter=3, leading=11.5, leftIndent=12,
                               textColor=colors.HexColor('#6b7280'), fontName='Helvetica-Oblique'))
    styles.add(ParagraphStyle(name='AnsText', fontSize=9.5, spaceAfter=9, leading=12.5, leftIndent=12,
                               textColor=colors.HexColor('#065f46'), fontName='Helvetica-Bold'))
    styles.add(ParagraphStyle(name='SectionCaption', fontSize=10, fontName='Helvetica-Bold',
                               textColor=colors.HexColor('#374151'), alignment=TA_CENTER, spaceBefore=6))

    story = []

    story.append(Paragraph("Facility Maturity Self-Assessment Report", styles['ReportTitle']))
    story.append(Paragraph(
        "Maturity evaluation across Physical, Digital and Organizational readiness layers",
        styles['ReportSubtitle']))

    meta_table = Table(
        [
            ["Respondent", meta['name'] or "-", "Facility", meta['facility'] or "-"],
            ["Role", meta['role'] or "-", "Date", datetime.now().strftime('%Y-%m-%d %H:%M')],
        ],
        colWidths=[70, 165, 55, 165],
    )
    meta_table.setStyle(TableStyle([
        ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
        ('FONTNAME', (2, 0), (2, -1), 'Helvetica-Bold'),
        ('FONTNAME', (1, 0), (1, -1), 'Helvetica'),
        ('FONTNAME', (3, 0), (3, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 9.5),
        ('TEXTCOLOR', (0, 0), (0, -1), colors.HexColor('#6b7280')),
        ('TEXTCOLOR', (2, 0), (2, -1), colors.HexColor('#6b7280')),
        ('TEXTCOLOR', (1, 0), (1, -1), colors.HexColor('#111827')),
        ('TEXTCOLOR', (3, 0), (3, -1), colors.HexColor('#111827')),
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#f9fafb')),
        ('BOX', (0, 0), (-1, -1), 0.75, colors.HexColor('#e5e7eb')),
        ('INNERGRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#e5e7eb')),
        ('TOPPADDING', (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('LEFTPADDING', (0, 0), (-1, -1), 8),
    ]))
    story.append(meta_table)
    story.append(Spacer(1, 4))

    grouped = group_widgets_by_layer_and_component(widget_store)

    answered = 0
    for item in widget_store.values():
        if item["widget"].value:
            answered = answered + 1
    total = len(widget_store)

    weighted_overall = scores.get("_overall", {}).get("weighted_overall_score")
    layer_weights = scores.get("_overall", {}).get("layer_weights", {})

    story.append(Spacer(1, 10))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor('#e5e7eb')))
    story.append(Paragraph("Technical Infrastructure Overview", styles['LayerHeading']))

    percent_done = 0
    if total > 0:
        percent_done = answered / total * 100
    story.append(Paragraph(
        "Completion: " + str(answered) + " of " + str(total) + " indicators answered (" + str(round(percent_done)) + "%).",
        styles['DescText']))

    if weighted_overall is not None:
        story.append(Paragraph(
            "Weighted overall maturity score: <b>" + format(weighted_overall, ".2f") + " / 4.00</b>",
            styles['LayerSubinfo']))
    story.append(Spacer(1, 6))
    overall_img = _scaled_image(radar_bytes.get("overall"), 3.6 * inch, 3.6 * inch)
    if overall_img:
        story.append(overall_img)
    story.append(PageBreak())

    for layer in grouped:
        comps = grouped[layer]
        accent_hex = LAYER_ACCENTS.get(layer, "#111827")
        accent = colors.HexColor(accent_hex)
        heading_style = ParagraphStyle('LayerHeading_' + layer, parent=styles['LayerHeading'], backColor=accent)
        story.append(Paragraph(layer + " Layer", heading_style))

        layer_score = scores.get(layer, {}).get("layer_score")
        layer_weight = layer_weights.get(layer, 0.5)

        info_bits = []
        if layer_score is not None:
            info_bits.append("Layer score: <b>" + format(layer_score, ".2f") + " / 4.00</b>")
        info_bits.append("Weight applied: <b>" + format(layer_weight, ".1f") + "x</b>")
        story.append(Paragraph("  &nbsp;|&nbsp;  ".join(info_bits), styles['LayerSubinfo']))

        layer_img = _scaled_image(radar_bytes["layers"].get(layer), 3.2 * inch, 3.2 * inch)
        if layer_img:
            story.append(Spacer(1, 4))
            story.append(layer_img)
            story.append(Paragraph(layer + " Layer - Component Scores (weighted)", styles['SectionCaption']))
            story.append(Spacer(1, 10))

        comp_weights = scores.get(layer, {}).get("component_weights", {})
        for comp in comps:
            items = comps[comp]
            comp_weight = comp_weights.get(comp, 0.5)
            comp_block = [
                Paragraph(comp, styles['CompHeading']),
                Paragraph("Weight applied: " + format(comp_weight, ".1f") + "x", styles['CompWeight']),
            ]
            for item in items:
                answer = item["widget"].value
                if not answer:
                    answer = "Not answered"
                comp_block.append(Paragraph(item['indicator'] + " - " + item['question'], styles['QText']))
                comp_block.append(Paragraph(item['description'], styles['DescText']))
                comp_block.append(Paragraph("Selected: " + answer, styles['AnsText']))
            story.append(KeepTogether(comp_block))

            comp_img = _scaled_image(radar_bytes["components"].get(layer + "::" + comp), 2.4 * inch, 2.4 * inch)
            if comp_img:
                story.append(comp_img)
                story.append(Spacer(1, 12))

        story.append(PageBreak())

    doc.build(story)
    return filename, answered, total

In [ ]:
def generate_csv(widget_store, base_filename):
    """Write one row per question: Layer, Infrastructure components, Readiness indicators,
    Description, Main Assessment Question, and the selected option as the last column."""
    filename = base_filename + ".csv"

    csv_file = open(filename, "w", newline="", encoding="utf-8")
    writer = csv.writer(csv_file)

    header = [
        "Layer",
        "Infrastructure components",
        "Readiness indicators",
        "Description",
        "Main Assessment Question",
        "Selected option",
    ]
    writer.writerow(header)

    for item in widget_store.values():
        answer = item["widget"].value
        if not answer:
            answer = "Not answered"
        row = [
            item["layer"],
            item["component"],
            item["indicator"],
            item["description"],
            item["question"],
            answer,
        ]
        writer.writerow(row)

    csv_file.close()
    return filename

In [ ]:
submit_btn = widgets.Button(
    description='Submit Assessment',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='240px')
)
submit_btn.style.button_color = '#2563eb'
submit_btn.style.font_weight = '600'
submit_btn.style.font_size = '16px'

output = widgets.Output()


def display_centered_image(img_bytes, max_width="420px"):
    b64 = base64.b64encode(img_bytes).decode("utf-8")
    html_text = (
        "<div style='display:flex; justify-content:center; margin:6px 0 18px;'>"
        "<img src='data:image/png;base64," + b64 + "' style='max-width:" + max_width + "; width:100%; height:auto;'>"
        "</div>"
    )
    display(HTML(html_text))


def on_submit_clicked(b):
    with output:
        clear_output()
        meta = {"name": resp_name.value, "facility": resp_facility.value, "role": resp_role.value}
        scores, radar_bytes = build_all_radars(ALL_WIDGETS, COMPONENT_WEIGHTS, LAYER_WEIGHTS)

        weighted_overall = scores.get("_overall", {}).get("weighted_overall_score")
        overall_text = ""
        if weighted_overall is not None:
            overall_text = " - Weighted score: <b>" + format(weighted_overall, ".2f") + " / 4.00</b>"

        title_html = (
            "<h3 style='font-family:Arial, sans-serif;color:#111827;text-align:center;'>"
            "Score Overview" + overall_text + "</h3>"
        )
        display(HTML(title_html))
        display_centered_image(radar_bytes["overall"], max_width="480px")

        for layer in radar_bytes["layers"]:
            display_centered_image(radar_bytes["layers"][layer], max_width="420px")
            for key in radar_bytes["components"]:
                if key.startswith(layer + "::"):
                    display_centered_image(radar_bytes["components"][key], max_width="320px")

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_name = make_safe_filename_part(meta.get('name'))
        base_filename = "maturity_assessment_" + safe_name + "_" + timestamp

        pdf_file, answered, total = generate_pdf(meta, ALL_WIDGETS, radar_bytes, scores, base_filename)
        csv_file = generate_csv(ALL_WIDGETS, base_filename)

        result_html = (
            "<div style='padding:20px;border-radius:10px;background-color:#d4edda;"
            "color:#155724;font-size:16px;font-family:Arial, sans-serif;'>"
            "<b>Thank you!</b><br>"
            + str(answered) + " of " + str(total) + " indicators answered.<br><br>"
            "PDF report saved as: <b>" + pdf_file + "</b><br>"
            "CSV report saved as: <b>" + csv_file + "</b>"
            "</div>"
        )
        display(HTML(result_html))


submit_btn.on_click(on_submit_clicked)

form_items = widgets.VBox([
    intro_header,
    resp_box,
    physical_section,
    digital_section,
    organizational_section,
    submit_btn,
    output
], layout=widgets.Layout(gap='4px', width='100%'))

styled_form = widgets.Box(
    [form_items],
    layout=widgets.Layout(
        border='solid 1px #d1d5db',
        padding='30px',
        width='95%',
        max_width='1600px',
        margin='24px auto',
        display='flex',
        flex_flow='column',
        align_items='stretch',
        background_color='#ffffff'
    )
)

display(styled_form)